In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Dataset
df = pd.read_csv("./data/customer_shopping_data.csv")

df.head()


,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon


In [2]:
# Check null data
df.isnull().sum()


invoice_no        0
customer_id       0
gender            0
age               0
category          0
quantity          0
price             0
payment_method    0
invoice_date      0
shopping_mall     0
dtype: int64

In [3]:
# Add data column
df['Total_Spend'] = df['quantity'] * df['price']

# Subdata - Shopping Mall & Payment Method vs Total Spend
df = df[['category','payment_method','shopping_mall','Total_Spend']]
# Reset index
df.reset_index(inplace=True, drop=True)

df.head()


,category,payment_method,shopping_mall,Total_Spend
0,Clothing,Credit Card,Kanyon,7502.00
1,Shoes,Debit Card,Forum Istanbul,5401.53
2,Clothing,Cash,Metrocity,300.08
3,Shoes,Credit Card,Metropol AVM,15004.25
4,Books,Cash,Kanyon,242.40


In [4]:
# Linear Regression Analysis

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
# Subset X and y variables
df_X = df[['category','payment_method','shopping_mall']]
df_y = df[['Total_Spend']]
X = pd.get_dummies(df_X, drop_first=True)

# Train 80% data, test 20% data 
X_train, X_test, y_train, y_test = train_test_split(X, df_y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("==== Model Evaluation Results ====")
print(f"Training Set Size : {X_train.shape[0]} samples")
print(f"Testing Set Size  : {X_test.shape[0]} samples")
print(f"RMSE              : {rmse:.2f} TL")
print(f"R-squared         : {r2:.4f}")
print("==================================")

==== Model Evaluation Results ====
Training Set Size : 79565 samples
Testing Set Size  : 19892 samples
RMSE              : 3037.82 TL
R-squared         : 0.4962


In [5]:
# Create DataFrame
coefficients = pd.DataFrame({
    'Feature': X.columns, 
    'Coefficient': model.coef_.flatten() 
})

print(coefficients.sort_values(by='Coefficient', ascending=False).to_string(index=False))

                        Feature  Coefficient
            category_Technology 11262.812200
                 category_Shoes  6459.570183
              category_Clothing  3148.826035
             category_Cosmetics   283.045279
                  category_Toys   226.645911
   shopping_mall_Forum Istanbul    32.101798
     shopping_mall_Istinye Park    18.928139
           shopping_mall_Kanyon    15.412975
     shopping_mall_Zorlu Center    12.945449
 shopping_mall_Mall of Istanbul    10.060266
     shopping_mall_Metropol AVM    -7.943557
     payment_method_Credit Card   -17.695017
   shopping_mall_Viaport Outlet   -19.287378
      payment_method_Debit Card   -25.749130
              category_Souvenir   -38.663317
shopping_mall_Emaar Square Mall   -63.174448
        shopping_mall_Metrocity   -74.665847
       category_Food & Beverage  -109.251421


Summary of Statistical and Machine Learning Insights:
1. **OLS Regression Analysis:** Using the full dataset for OLS analysis yielded an R-squared of 0.4, demonstrating that the selected features explain 48% of the variance in Total_Spend. Specifically, Product Category stands out as the dominant factor with a highly significant relationship (p < 0.001). Technology has the strongest positive impact (adding approx. 11,410 TL per transaction), closely followed by Shoes and Clothing.

2. **Linear Regression (Machine Learning):** To evaluate predictive performance, the dataset was split into 80% for training and 20% for testing. The model achieved a robust R**-squared of 0.4962** on unseen test data, with a Root Mean Squared Error (RMSE) of 3,037.82 TL. Consistent with the OLS findings, Technology, Shoes, and Clothing emerged as the top three categorical drivers with the highest positive coefficients.

3. **Strategic Takeaway**: Both methods perfectly validate each other: transaction sizes are fundamentally driven by what customers buy (high-value categories) rather than where they shop (Shopping Mall) or how they pay (Payment Method). Marketing efforts should focus on cross-selling Technology and Shoes to maximize basket value.